# FIFA World Cup 2026 — Analytic Task
## Shot-on-Target Accuracy by Position (Forwards vs Midfielders)

**Analytic question:** Is there a significant difference in shot-on-target accuracy (SoT%) between forward (FW) and midfielder (MF) players at the 2026 FIFA World Cup? Players who did not attempt any shots are excluded.

**Reasoning:** Forwards are expected to have higher shot accuracy because their main role is finishing chances close to goal, often in good scoring positions. Midfielders take a wider range of shots, including from further out, which may lower their accuracy.

**Population:** Forward and midfielder players who attempted at least one shot at the 2026 World Cup. Players with a mixed position label (for example "FW,MF") are excluded because they cannot be placed into a single group.

**Test:** A two-sample t-test, since we are comparing two independent groups (forwards and midfielders).

**Data source:** FBref.com — 2026 World Cup player statistics, cleaned and combined into `players_clean.csv` by the team.

## 1. Setup

In [ ]:
import pandas as pd
import numpy as np
from scipy import stats
import matplotlib.pyplot as plt

DATA_FILE = "players_clean.csv"
SAMPLE_SIZE = 60
RANDOM_SEED = 42

np.random.seed(RANDOM_SEED)

## 2. Data Wrangling

The raw file contains every player at the tournament, including defenders and goalkeepers, and some players with mixed position labels. We keep only forwards and midfielders with a single, clear position, and only players who attempted at least one shot (SoT% is undefined for 0 shots).

In [ ]:
data = pd.read_csv(DATA_FILE)
print(f"Total players loaded: {len(data)}")
print(data["pos"].value_counts())

# keep only forwards and midfielders (drop mixed labels like "FW,MF")
df = data[(data["pos"] == "FW") | (data["pos"] == "MF")].copy()

# keep only players who took at least 1 shot
df = df[df["shots"] > 0].copy()

# keep only the columns needed, and rename the long column
df = df[["player", "pos", "squad", "shots", "shots_on_target", "shooting_standard_sot%"]]
df = df.rename(columns={"shooting_standard_sot%": "sot_percent"})

print(f"\nPopulation after wrangling: {len(df)} players "
      f"({(df['pos']=='FW').sum()} FW, {(df['pos']=='MF').sum()} MF)")
df.head()

## 3. Data Preparation & Sampling

A simple random sample of 60 players is drawn from each group. A sample of 60 per group is well above the general rule of at least 30 observations needed for the Central Limit Theorem to apply, which supports using a t-test on the sample means.

In [ ]:
forwards_pop   = df[df["pos"] == "FW"]
midfielders_pop = df[df["pos"] == "MF"]

forwards_sample   = forwards_pop.sample(n=SAMPLE_SIZE, random_state=RANDOM_SEED)
midfielders_sample = midfielders_pop.sample(n=SAMPLE_SIZE, random_state=RANDOM_SEED)

sample = pd.concat([forwards_sample, midfielders_sample], ignore_index=True)

print(f"Sample drawn: n={len(sample)} "
      f"({(sample['pos']=='FW').sum()} FW, {(sample['pos']=='MF').sum()} MF)")
sample.head()

## 4. Descriptive Statistics

In [ ]:
desc = sample.groupby("pos")["sot_percent"].agg(
    ["count", "mean", "std", "median", lambda x: x.quantile(.25), lambda x: x.quantile(.75)])
desc.columns = ["n", "mean", "sd", "median", "Q1", "Q3"]
desc.round(3)

## 5. Inferential Statistics — Confidence Interval

A 95% confidence interval is calculated for each group's mean SoT%, showing the range where the true population mean is likely to fall.

In [ ]:
fw = sample.loc[sample["pos"] == "FW", "sot_percent"]
mf = sample.loc[sample["pos"] == "MF", "sot_percent"]

for label, group in [("Forwards", fw), ("Midfielders", mf)]:
    n = len(group)
    mean, se = group.mean(), group.std(ddof=1) / np.sqrt(n)
    ci_low, ci_high = stats.t.interval(0.95, df=n - 1, loc=mean, scale=se)
    print(f"{label}: mean = {mean:.3f}  95% CI = [{ci_low:.3f}, {ci_high:.3f}]  (n={n})")

## 6. Inferential Statistics — Two-Sample t-Test

Welch's version of the two-sample t-test is used because it does not assume the two groups have equal variance, which is a safer default when comparing two independent samples.

- H0 (null hypothesis): forwards and midfielders have the same average shot accuracy
- H1 (alternative hypothesis): the average shot accuracy is different between the two groups

In [ ]:
t_stat, p_val = stats.ttest_ind(fw, mf, equal_var=False)

print(f"Forwards:    n={len(fw)}, mean={fw.mean():.3f}, sd={fw.std(ddof=1):.3f}")
print(f"Midfielders: n={len(mf)}, mean={mf.mean():.3f}, sd={mf.std(ddof=1):.3f}")
print(f"t = {t_stat:.3f}, p = {p_val:.4f}")

verdict = "statistically significant" if p_val < 0.05 else "not statistically significant"
print(f"--> Difference is {verdict} at alpha = 0.05")

## 7. Visualisation

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4.5))
groups = ["Forwards", "Midfielders"]
means = [fw.mean(), mf.mean()]
errs = [fw.std(ddof=1) / np.sqrt(len(fw)) * stats.t.ppf(0.975, len(fw) - 1),
        mf.std(ddof=1) / np.sqrt(len(mf)) * stats.t.ppf(0.975, len(mf) - 1)]

ax.bar(groups, means, yerr=errs, capsize=8, color=["#d9a441", "#1b5e42"])
ax.set_ylabel("Shot-on-Target Accuracy (%)")
ax.set_title("2026 World Cup: Shot Accuracy\nForwards vs Midfielders (95% CI)")
plt.tight_layout()
plt.savefig("sot_by_position.png", dpi=150)
plt.show()

## 8. Findings

Forwards had a noticeably higher shot-on-target accuracy than midfielders in the sample: 40.65% compared with 26.95%, a difference of about 13.7 percentage points. The 95% confidence intervals for the two groups (roughly [32.8%, 48.5%] for forwards and [19.4%, 34.5%] for midfielders) do not fully overlap, which already suggests a real difference.

The two-sample t-test gave **t = 2.516** and **p = 0.0132**. Since p < 0.05, the null hypothesis is rejected — the difference in shot accuracy between forwards and midfielders is unlikely to be due to random chance alone.

This fits with what we know about the two positions: forwards are usually positioned closer to goal and their main role is finishing chances, while midfielders take a wider variety of shots, including from further out, which would naturally lower their accuracy.

**Limitations:** Players with a mixed position label (for example "FW,MF") were excluded because they could not be placed into a single group, and players who never attempted a shot were excluded because their SoT% cannot be calculated. The results apply only to players with a single, clear position who took at least one shot.